In [ ]:
%%bash
set -euo pipefail
REPO_URL="https://github.com/dangkhoa2016/KerasHub-TranslateGemma-4B-IT-Kaggle-GPU-T4x2-Text-Vision.git"
PROJECT="/kaggle/working/KerasHub-TranslateGemma-4B-IT-Kaggle-GPU-T4x2-Text-Vision"

if [[ -d "$PROJECT/.git" ]]; then
  echo "Refreshing existing repository checkout..."
  git -C "$PROJECT" fetch --depth 1 origin main
  git -C "$PROJECT" reset --hard origin/main
else
  rm -rf "$PROJECT"
  git clone --depth 1 "$REPO_URL" "$PROJECT"
fi

cd "$PROJECT"
cp .env.example .env
printf 'Repository ready: %s\n' "$PROJECT"


# TranslateGemma 4B IT — Kaggle T4x2 — Text + Vision

This notebook runs the public GitHub repository directly on a Kaggle **GPU T4 x2** session.

Before continuing:

- select **GPU T4 x2** as the accelerator;
- enable **Internet** for the repository checkout and optional dependency/tunnel download;
- attach the Keras TranslateGemma model containing the `translategemma_4b_it` preset.

The first cell above clones or refreshes the repository. The remaining cells validate the environment, run unit tests, start two multimodal workers, test text and image translation, and optionally run performance benchmarks.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request

WORK = Path("/kaggle/working")
PROJECT = WORK / "KerasHub-TranslateGemma-4B-IT-Kaggle-GPU-T4x2-Text-Vision"
READY_TIMEOUT = 1800

if not PROJECT.is_dir():
    raise FileNotFoundError(f"Repository checkout not found: {PROJECT}")


def run(args, *, cwd=PROJECT, env=None, check=True):
    print("$", " ".join(map(str, args)))
    return subprocess.run(
        args,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=check,
        text=True,
    )


def simple_env(path):
    values = {}
    for raw in path.read_text(encoding="utf-8").splitlines():
        raw = raw.strip()
        if raw and not raw.startswith("#") and "=" in raw:
            key, value = raw.split("=", 1)
            values[key] = value.strip().strip('"').strip("'")
    return values


def dir_bytes(path):
    if not path.is_dir():
        return 0
    total = 0
    for child in path.rglob("*"):
        try:
            if child.is_file():
                total += child.stat().st_size
        except OSError:
            pass
    return total

print("PROJECT =", PROJECT)


## 1. T4x2 and configuration preflight

Validate that Kaggle exposes two Tesla T4 GPUs and that the repository configuration is set for two multimodal workers. You can set `CLEAR_JAX_CACHE_BEFORE_RUN=True` for an explicit cold-cache experiment.


In [ ]:
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Enable Accelerator = GPU T4 x2 before running this notebook.")

gpu_rows = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=index,name,memory.total,memory.free",
        "--format=csv,noheader,nounits",
    ],
    text=True,
    capture_output=True,
    check=True,
).stdout.strip().splitlines()
print("\n".join(gpu_rows))
if len(gpu_rows) < 2 or not all("t4" in row.lower() for row in gpu_rows[:2]):
    raise RuntimeError("This notebook requires a Kaggle GPU T4 x2 session.")

cfg = simple_env(PROJECT / ".env")
required = {
    "MAX_GPU_WORKERS": "2",
    "GPU_IDS": "0,1",
    "VISION_ENABLED": "true",
    "JAX_PREALLOCATE": "true",
    "JAX_MEM_FRACTION": "0.97",
    "GENERATION_BUCKETING": "true",
    "GENERATION_LENGTH_BUCKETS": "256,512,1024,1536,2048",
    "WARMUP_TEXT_BUCKETS": "256",
    "WARMUP_VISION_BUCKETS": "512",
    "WORKER_START_MODE": "auto",
}
invalid = {key: (cfg.get(key), expected) for key, expected in required.items() if cfg.get(key) != expected}
if invalid:
    raise RuntimeError(f"Repository configuration is not the expected T4x2 profile: {invalid}")
print("T4x2 configuration OK:", required)

cache_path = Path(cfg.get("JAX_COMPILATION_CACHE_DIR", "")).expanduser()
print("JAX cache before run:", cache_path, dir_bytes(cache_path), "bytes")

CLEAR_JAX_CACHE_BEFORE_RUN = False
if CLEAR_JAX_CACHE_BEFORE_RUN and cache_path.is_dir():
    if cache_path.name != "translategemma-jax":
        raise RuntimeError(f"Refusing to clear an unexpected cache path: {cache_path}")
    shutil.rmtree(cache_path)
    print("Cleared the dedicated TranslateGemma JAX cache.")


## 2. Stop any previous runtime

This makes notebook re-runs deterministic without restoring any external session state.


In [ ]:
run(["bash", "scripts/stop_tunnel.sh"], check=False)
run(["bash", "scripts/stop.sh"], check=False)


## 3. Setup dependencies and run unit tests

Kaggle normally provides the CUDA-enabled JAX/JAXLIB pair. The setup script installs/checks the remaining Python dependencies without intentionally replacing Kaggle's GPU-enabled JAX stack.


In [ ]:
setup_env = os.environ.copy()
setup_env["INSTALL_PYTHON_DEPS"] = "1"
run(["bash", "scripts/setup.sh"], env=setup_env)

test_env = os.environ.copy()
test_env["PYTHONPATH"] = "src"
run(["python3", "-m", "unittest", "discover", "-s", "tests", "-v"], env=test_env)


## 4. Start the API coordinator and two multimodal GPU workers


In [ ]:
run(["bash", "scripts/start.sh"])


## 5. Wait for 2/2 multimodal workers

The detailed readiness response confirms that both workers are ready and that vision mode is enabled on each worker.


In [ ]:
cfg = simple_env(PROJECT / ".env")
base_url = f"http://127.0.0.1:{cfg.get('PORT', '7860')}"
key_file = PROJECT / "data/api_key.txt"
deadline = time.monotonic() + READY_TIMEOUT
health = None
last_summary = None

while time.monotonic() < deadline:
    if key_file.exists() and key_file.read_text(encoding="utf-8").strip():
        key = key_file.read_text(encoding="utf-8").strip()
        request = urllib.request.Request(
            base_url + "/health/ready?all=1&details=1",
            headers={"Authorization": f"Bearer {key}"},
        )
        try:
            with urllib.request.urlopen(request, timeout=10) as response:
                data = json.loads(response.read().decode())
                workers = data.get("workers") or []
                summary = (
                    data.get("state"),
                    data.get("ready_workers"),
                    data.get("expected_workers"),
                    [
                        (
                            worker.get("worker_id"),
                            worker.get("state"),
                            (worker.get("metadata") or {}).get("vision_enabled"),
                        )
                        for worker in workers
                    ],
                )
                if summary != last_summary:
                    print(summary)
                    last_summary = summary
                if (
                    data.get("ready_workers") == 2
                    and data.get("expected_workers") == 2
                    and len(workers) == 2
                    and all((worker.get("metadata") or {}).get("vision_enabled") is True for worker in workers)
                ):
                    health = data
                    break
        except (urllib.error.URLError, urllib.error.HTTPError):
            pass
    time.sleep(5)

if health is None:
    run(["bash", "scripts/status.sh"], check=False)
    run(
        [
            "bash",
            "-lc",
            "tail -n 120 log/server.log 2>/dev/null || true; "
            "tail -n 120 log/worker-gpu-0.log 2>/dev/null || true; "
            "tail -n 120 log/worker-gpu-1.log 2>/dev/null || true",
        ],
        check=False,
    )
    raise TimeoutError("The server did not reach 2/2 vision-enabled workers before the timeout.")

print("PASS: 2/2 multimodal vision workers are ready")


## 6. Inspect warm-up and persistent compilation-cache metadata


In [ ]:
for worker in health.get("workers") or []:
    metadata = worker.get("metadata") or {}
    print(
        worker.get("worker_id"),
        json.dumps(
            {
                "gpu_id": worker.get("gpu_id"),
                "dtype": metadata.get("dtype"),
                "generation_bucketing": metadata.get("generation_bucketing"),
                "generation_length_buckets": metadata.get("generation_length_buckets"),
                "warmup": metadata.get("warmup"),
                "load_seconds": metadata.get("load_seconds"),
                "compilation_cache_dir": metadata.get("compilation_cache_dir"),
            },
            indent=2,
        ),
    )
print("JAX cache after 2/2 ready:", cache_path, dir_bytes(cache_path), "bytes")


## 7. Text translation smoke test

This uses the same multimodal workers used by the image endpoint.


In [ ]:
run(["bash", "scripts/test.sh", "data/input.example.txt"])
print("PASS: text translation smoke test")


## 8. Vision OCR + translation smoke test


In [ ]:
run(["bash", "scripts/test_vision.sh", "assets/sample-image-with-text.jpg"])
print("PASS: vision translation smoke test")


## 9. PRIME + HOT two-request T4x2 concurrency benchmark

The benchmark sends a pair of concurrent requests twice. The PRIME phase can include a shape/cache miss; the HOT phase immediately repeats the same workload. The benchmark verifies that the two jobs use different GPU workers.


In [ ]:
run(["bash", "scripts/test_concurrency.sh"])
print("PASS: T4x2 concurrent inference")


## 10. Optional BF16 vs FP16 benchmark

The validated reference run found very similar hot-phase performance, so BF16 is the default. Enable this only when you want to re-measure the current Kaggle image/model combination.


In [ ]:
RUN_DTYPE_BENCHMARK = False
if RUN_DTYPE_BENCHMARK:
    dtype_env = os.environ.copy()
    dtype_env["RUN_DTYPE_BENCHMARK"] = "1"
    run(["bash", "scripts/benchmark_dtype.sh"], env=dtype_env)
else:
    print("SKIP dtype benchmark (set RUN_DTYPE_BENCHMARK=True to enable).")


## 11. Optional startup/cache benchmark

This benchmark compares cold-cache staggered startup, warm-cache staggered startup, and warm-cache parallel startup, then restores the normal server configuration.


In [ ]:
RUN_STARTUP_BENCHMARK = False
if RUN_STARTUP_BENCHMARK:
    startup_env = os.environ.copy()
    startup_env["RUN_STARTUP_BENCHMARK"] = "1"
    run(["bash", "scripts/benchmark_startup.sh"], env=startup_env)
else:
    print("SKIP startup benchmark (set RUN_STARTUP_BENCHMARK=True to enable).")


## 12. Optional temporary public tunnel

Keep API authentication enabled. The generated Cloudflare Quick Tunnel URL is temporary and is written only to the local runtime working tree.


In [ ]:
START_PUBLIC_TUNNEL = False
if START_PUBLIC_TUNNEL:
    run(["bash", "scripts/run_tunnel.sh"])
else:
    print("SKIP public tunnel (set START_PUBLIC_TUNNEL=True to enable).")


## 13. Final status


In [ ]:
run(["bash", "scripts/status.sh"], check=False)
print("Completed the Kaggle T4x2 text + vision validation workflow.")
